# Group Isoforms into C-terminal and not C-terminal (Internal and N-terminal)
In the final database, we will add the entire new C-terminus for the isoforms that have a new C-terminus (can include several tryptic cleavage sites). For isoforms sharing a C-terminus with the canonical form, we will perform in-silico tryptic digestion and only add a unique fragment to the database (if we can find one).

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import re

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Database"):
    print("WARNING: The working directory is not set to the '02_Isoform_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_Isoform_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
mapping_dir = "../01_UniProt/data/mapping/"
unique_dir = "../01_UniProt/data/unique/"

## Read in fasta files and mapping

In [ ]:
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))
fasta_canonical_unique = os.path.join(unique_dir, "unique_canonical.fasta")

iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

# Define unique isoform peptides 

In [ ]:
def digest_protein_with_coordinates(seq):
    """
    Performs an in silico tryptic digestion of a full protein.
    Returns a list of dictionaries containing the peptide string, 
    its 1-based start coordinate, and its 1-based end coordinate.
    """
    if not seq:
        return []
    
    cuts = [0]
    for i in range(len(seq) - 1):
        if seq[i] in ('K', 'R') and seq[i+1] != 'P':
            cuts.append(i + 1)
            
    if seq[-1] in ('K', 'R'):
        cuts.append(len(seq))
        
    if cuts[-1] != len(seq):
        cuts.append(len(seq))
        
    peptides_with_coords = []
    for start, end in zip(cuts[:-1], cuts[1:]):
        peptides_with_coords.append({
            'peptide': seq[start:end],
            'start': start + 1,  # Convert to 1-based index
            'end': end
        })
    return peptides_with_coords

In [ ]:
def extract_unique_isoform_peptides(mapping_df, seq_dict, min_len=7, max_len=52):
    """
    Finds ALL true unique tryptic peptides for all isoforms across the entire sequence.
    Classifies location types accurately, enforcing that a peptide is only labeled 
    as 'Alternative C-Terminus' if the actual physical C-terminus has changed.
    """
    results_list = []
    processed_isoforms = set()

    for _, row in mapping_df.iterrows():
        c_id = row['Entry']
        i_id = row['Isoform_ID']

        if i_id in processed_isoforms:
            continue
            
        c_seq = seq_dict.get(c_id)
        i_seq = seq_dict.get(i_id)
        if not c_seq or not i_seq: 
            continue

        # 1. Get full-length digests for both proteins with structural positions
        c_digest = digest_protein_with_coordinates(c_seq)
        i_digest = digest_protein_with_coordinates(i_seq)
        
        if not c_digest or not i_digest:
            continue
        
        # 2. Build a rapid lookup set of every peptide existing in the canonical sequence
        canonical_peptide_pool = set(p['peptide'] for p in c_digest)

        # 3. CRITICAL CHECK: Determine if the true physical C-termini are actually different
        true_c_term_has_changed = (i_digest[-1]['peptide'] != c_digest[-1]['peptide'])

        # 4. Sweep through all isoform peptides globally
        for p_info in i_digest:
            pep = p_info['peptide']
            pep_len = len(pep)
            
            # Condition A: Peptide must be completely absent from the canonical pool
            if pep not in canonical_peptide_pool:
                
                # Condition B: Must fit your exact mass spectrometry length window
                if min_len <= pep_len <= max_len:
                    
                    # 5. Classify Location Type based on structural rules
                    is_at_isoform_start = (p_info['start'] == 1)
                    is_at_isoform_end = (p_info['end'] == len(i_seq))
                    
                    if is_at_isoform_start:
                        loc_type = "N_term"
                    # Only label C-terminal if the unique peptide touches the end AND the end is genuinely changed
                    elif is_at_isoform_end and true_c_term_has_changed:
                        loc_type = "C_term"
                    else:
                        # Upstream unique fragments or unique fragments where the c-terminus 
                        # resolves back to the canonical tail are cleanly flagged here.
                        loc_type = "Internal"

                    results_list.append({
                        'Isoform_ID': i_id,
                        'Canonical_ID': c_id,
                        'Location_Type': loc_type,
                        'Captured_Peptide': pep,
                        'Peptide_Length': pep_len,
                        'Peptide_Coordinates_In_Isoform': f"{p_info['start']}-{p_info['end']}"
                    })
                        
        processed_isoforms.add(i_id)
            
    return pd.DataFrame(results_list)

In [ ]:
# --- Execution ---
seq_lookup = full_proteome_unique.set_index('ID')['seq'].to_dict()
df_iso_peptides = extract_unique_isoform_peptides(iso_canonical_mapping_flat, seq_lookup, min_len=7, max_len=52)

In [ ]:
df_iso_peptides

In [ ]:
duplicates = df_iso_peptides[df_iso_peptides.duplicated(subset=['Captured_Peptide'], keep=False)]

In [ ]:
df_peptides_unique = df_iso_peptides[
    ~df_iso_peptides['Captured_Peptide'].isin(duplicates['Captured_Peptide'].unique())
]
df_peptides_unique

In [ ]:
len(df_peptides_unique["Isoform_ID"].unique())

# Fasta Files for Database
The fasta file contains all unique tryptic peptides that can identify an isoform. The header contains the coordinates of the peptide in the isoform, so if there are several unique peptides in an isoform they are distinguishable by their position.

In [ ]:
final_records = []

for _, row in df_peptides_unique.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Captured_Peptide']
    location = row["Location_Type"]  
    coordinates = row["Peptide_Coordinates_In_Isoform"]
    
    # Add coordinates to the location part in the header
    header_id = f"sp|{iso_id}|{location}_{coordinates}"
    
    # Construct the SeqRecord
    rec = SeqRecord(Seq(pep_seq), id=header_id, description="")
    final_records.append(rec)


# Write to file
with open("Isoform_Database_only_iso.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")